### Bringing in document group data to make reprint group id metadata
Followed by merging reprint document groups with large metadata sheet

In [3]:
import pandas as pd

In [4]:
# ---------- CONFIG ----------
INPUT_PATH = "document_groups.csv"
OUTPUT_PATH = "document_groups_labeled.csv"

# 1. Load data
df = pd.read_csv(INPUT_PATH)

# 2. Normalize blanks to NaN in the key columns
cols = ["article_id_parent", "article_id_direct", "article_id_truncated", "article_id_paraphrase"]
for col in cols:
    if col in df.columns:
        df[col] = df[col].replace(r"^\s*$", pd.NA, regex=True)

# 3. Forward-fill the parent so every row knows its original
df["group_parent"] = df["article_id_parent"].ffill()

# If there are any leading rows without a parent, you can drop them:
df = df[df["group_parent"].notna()].copy()

# Helper: base group_id_type (original + _reprint)
df["group_id_type"] = df["group_parent"].astype(str) + "_reprint"

# 4. Build separate DataFrames for original, direct, truncated

# a) Originals (just the parent rows, once each)
orig = (
    df.loc[df["article_id_parent"].notna(), ["article_id_parent", "group_id_type"]]
      .rename(columns={"article_id_parent": "article_id"})
)
orig["reprint_type"] = "original"

# b) Direct reprints
direct = (
    df.loc[df["article_id_direct"].notna(), ["article_id_direct", "group_id_type"]]
      .rename(columns={"article_id_direct": "article_id"})
)
direct["reprint_type"] = "direct"

# c) Truncated reprints
trunc = (
    df.loc[df["article_id_truncated"].notna(), ["article_id_truncated", "group_id_type"]]
      .rename(columns={"article_id_truncated": "article_id"})
)
trunc["reprint_type"] = "truncated"

# 5. Combine them; paraphrase column is ignored by design
out = pd.concat([orig, direct, trunc], ignore_index=True)

# 6. Drop duplicates if same article_id appears multiple times for any reason
out = out.drop_duplicates(subset=["article_id", "group_id_type", "reprint_type"])

# 7. Save final long-form table
out.to_csv(OUTPUT_PATH, index=False)

print(f"Saved reprint mapping to {OUTPUT_PATH}")


Saved reprint mapping to document_groups_labeled.csv


In [5]:
import pandas as pd

# ---------- CONFIG ----------
META_PATH = "metadata_with_tropes_transcripts.csv"
REPRINTS_PATH = "document_groups_labeled.csv"
OUTPUT_PATH = "complete_metadata_images_tropes_reprints_transcripts.csv"

# 1. Load both sheets
meta = pd.read_csv(META_PATH)
reprints = pd.read_csv(REPRINTS_PATH)

# 2. Normalize article_id (strip whitespace, force string)
meta["article_id"] = meta["article_id"].astype(str).str.strip()
reprints["article_id"] = reprints["article_id"].astype(str).str.strip()

# 3. Rename group_id_type -> group_reprint_id to match your metadata column naming
if "group_id_type" in reprints.columns:
    reprints = reprints.rename(columns={"group_id_type": "group_reprint_id"})

# 4. If the metadata already has empty group_reprint_id / reprint_type columns,
#    drop them so we can replace them cleanly from reprints
meta = meta.drop(columns=["group_reprint_id", "reprint_type"], errors="ignore")

# 5. Merge: LEFT JOIN so every image row stays,
#    and we add group_reprint_id + reprint_type wherever article_id matches
meta_merged = meta.merge(
    reprints[["article_id", "group_reprint_id", "reprint_type"]],
    on="article_id",
    how="left"
)

# 6. Optional: fill NaN in these new columns with empty strings (instead of <NA>)
meta_merged["group_reprint_id"] = meta_merged["group_reprint_id"].fillna("")
meta_merged["reprint_type"] = meta_merged["reprint_type"].fillna("")

# 7. Save final sheet
meta_merged.to_csv(OUTPUT_PATH, index=False)

print(f"Saved updated metadata with reprint info to: {OUTPUT_PATH}")


Saved updated metadata with reprint info to: complete_metadata_images_tropes_reprints_transcripts.csv


In [8]:
# sanity check

meta_merged[meta_merged["article_id"] == "Stuart1878"][
    ["object_id", "article_id", "group_reprint_id", "reprint_type"]
]


,object_id,article_id,group_reprint_id,reprint_type
2903,Stuart1878_Original,Stuart1878,Stuart1878_reprint,original
2904,Stuart1878_img1,Stuart1878,Stuart1878_reprint,original
2905,Stuart1878_img2,Stuart1878,Stuart1878_reprint,original
2906,Stuart1878_img3,Stuart1878,Stuart1878_reprint,original
2907,Stuart1878_img4,Stuart1878,Stuart1878_reprint,original
2908,Stuart1878_img5,Stuart1878,Stuart1878_reprint,original
2909,Stuart1878_img6,Stuart1878,Stuart1878_reprint,original
2910,Stuart1878_img7,Stuart1878,Stuart1878_reprint,original
2911,Stuart1878_img8,Stuart1878,Stuart1878_reprint,original
2912,Stuart1878_img9,Stuart1878,Stuart1878_reprint,original
